In [4]:
import pandas as pd
import numpy as np
import re

# ==========================
# LOAD DATA
# ==========================

file="../9. Near Miss - 2014.xlsx"

xls=pd.ExcelFile(file)

print("Sheets:")
print(xls.sheet_names)

sheet=xls.sheet_names[0]

df=pd.read_excel(
    file,
    sheet_name=sheet,
    dtype=str
)

original_shape=df.shape

print("\nOriginal Shape:",original_shape)

# ==========================
# CLEAN COLUMN NAMES
# ==========================

df.columns=(
    df.columns
    .astype(str)
    .str.strip()
    .str.replace("\n"," ",regex=False)
    .str.replace(r"\s+","_",regex=True)
)

# ==========================
# STANDARDIZE EMPTY VALUES
# ==========================

df=df.replace(
    [
        "NA",
        "N/A",
        "na",
        "n/a",
        "NULL",
        "null",
        "nan",
        "Not Mentioned",
        "not mentioned",
        "NOT MENTIONED"
    ],
    np.nan
)

# ==========================
# REMOVE EMPTY COLUMNS
# ==========================

drop_cols=[]

for col in df.columns:

    empty=(
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .all()
    )

    if empty:
        drop_cols.append(col)

df=df.drop(
    columns=drop_cols,
    errors="ignore"
)

print("\nRemoved Empty Columns:")
print(drop_cols)

# ==========================
# FIND DESCRIPTION COLUMN
# ==========================

desc_col=None

for col in df.columns:

    if "description" in col.lower():

        desc_col=col
        break

# ==========================
# REMOVE EMPTY DESCRIPTION
# ==========================

if desc_col:

    before=len(df)

    df=df[
        df[desc_col]
        .fillna("")
        .astype(str)
        .str.strip()
        !=""
    ]

    print(
        "\nRows Removed (Description Empty):",
        before-len(df)
    )

# ==========================
# SAFE DATE FORMAT
# ==========================

required_dates=[

    "date_of_the_near_miss",
    "date_corrective_action_completed",
    "date_investigation_commenced",
    "target_date",
    "date_of_investigation_completion"

]

date_cols=[]

for col in df.columns:

    c=col.lower()

    if c in required_dates:

        date_cols.append(col)

def clean_date(v):

    if pd.isna(v):
        return np.nan

    v=str(v).strip()

    if v=="":
        return np.nan

    if v.lower() in [
        "na",
        "n/a",
        "not mentioned"
    ]:
        return np.nan

    # month only → keep
    if re.match(
        r"^[A-Za-z]{3}$",
        v
    ):
        return v

    # already formatted → keep
    if re.match(
        r"^\d{2}-[A-Za-z]{3}-\d{2}$",
        v
    ):
        return v

    # original dd-mm-yyyy → parse safely
    if re.match(
        r"^\d{1,2}-\d{1,2}-\d{4}$",
        v
    ):

        try:

            d=pd.to_datetime(
                v,
                dayfirst=True,
                errors="coerce"
            )

            if pd.notna(d):

                return d.strftime(
                    "%d-%b-%y"
                )

        except:
            pass

    try:

        d=pd.to_datetime(
            v,
            errors="coerce"
        )

        if pd.notna(d):

            return d.strftime(
                "%d-%b-%y"
            )

    except:
        pass

    return v


for col in date_cols:

    df[col]=(
        df[col]
        .apply(clean_date)
    )

print("\nFormatted Date Columns:")
print(date_cols)

# ==========================
# REMOVE DUPLICATES
# ==========================

duplicates=df.duplicated().sum()

df=df.drop_duplicates()

print("\nDuplicates Removed:",duplicates)

# ==========================
# ADD + RESET SERIAL NUMBER
# ==========================

# remove old SI column if accidentally exists
existing_si=[]

for col in df.columns:

    c=(
        str(col)
        .lower()
        .replace("_","")
        .replace(":","")
        .replace(".","")
        .replace(" ","")
    )

    if c in [
        "sino",
        "slno",
        "serialno"
    ]:
        existing_si.append(col)

if existing_si:

    df=df.drop(
        columns=existing_si,
        errors="ignore"
    )

# reset index first
df=df.reset_index(
    drop=True
)

# insert new SI_No at first column
df.insert(
    0,
    "SI_No",
    np.arange(
        1,
        len(df)+1
    )
)

print("\nSI_No created and reordered")
# ==========================
# MISSING SUMMARY
# ==========================

missing=(
    df
    .replace("",np.nan)
    .isna()
    .sum()
)

missing=missing[
    missing>0
]

print("\nMissing Values Summary:")
print(missing)

# ==========================
# FINAL REPORT
# ==========================

print("\nFinal Shape:")
print(df.shape)

print("\nColumns After Cleaning:")
print(list(df.columns))

# ==========================
# EXPORT
# ==========================

output="cleaned_9_Near_Miss_2014.xlsx"

df.to_excel(
    output,
    index=False
)

print("\nSaved:",output)

Sheets:
['Near Miss']

Original Shape: (895, 22)

Removed Empty Columns:
['PIC']

Rows Removed (Description Empty): 0


C:\Users\vinyt\AppData\Local\Temp\ipykernel_29232\1118247918.py:44: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df=df.replace(



Formatted Date Columns:
['Date_of_the_Near_Miss', 'Date_Corrective_action_completed', 'Date_Investigation_Commenced', 'Target_Date', 'Date_of_Investigation_Completion']

Duplicates Removed: 0

SI_No created and reordered

Missing Values Summary:
Ref.No.                                       1
Type_of_Ship                                  8
Incident_Category_(Potential)                 8
Location_of_the_Unsafe_act_/_condition       13
Possible_Consequence                         49
Probability_of_Reoccurence                    7
Cause_Analysis                                4
Counter_measure_to_prevent_recurrence       252
Suggested_Corrective_Action_(Office_Use)    274
Date_Corrective_action_completed            613
Area_of_Concern                             187
Date_Investigation_Commenced                643
Target_Date                                 640
Date_of_Investigation_Completion            652
Extension_Remarks                           893
Extended_To                      